# Luật kết hợp: dịch vụ/hành vi liên hệ với Churn

Notebook này **gọi lại code chuẩn trong `src`**, không tự tính lại bằng logic khác. Mỗi dòng khách hàng được đổi thành một giao dịch, sau đó Apriori và FP-Growth tìm luật có vế phải `Churn=Yes`.

Độ đo cần đọc: Support (tỷ lệ xuất hiện), Confidence (độ tin cậy) và Lift. Lift > 1 nghĩa là tổ hợp ở vế trái xuất hiện cùng Churn=Yes nhiều hơn mức ngẫu nhiên kỳ vọng.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

# Cho phep chay notebook khi dang mo trong thu muc notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.association_data import build_transaction_table
from src.association_rules import mine_churn_rules, save_top_rules_plot

DATA_PATH = PROJECT_ROOT / 'data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
REPORTS_DIR = PROJECT_ROOT / 'reports'
MIN_SUPPORT = 0.05
MIN_CONFIDENCE = 0.40
MIN_LIFT = 1.0
MAX_LEN = 3

## 1. Tạo dữ liệu giao dịch

`customerID` bị loại. Hai cột số `tenure` và `MonthlyCharges` được chia khoảng để trở thành item có thể khai phá luật.

In [ ]:
raw = pd.read_csv(DATA_PATH)
transactions = build_transaction_table(raw)
print(f'Dữ liệu gốc: {raw.shape[0]:,} khách hàng, {raw.shape[1]} cột')
print(f'Ma trận giao dịch: {transactions.shape[0]:,} giao dịch, {transactions.shape[1]} item')
transactions.iloc[:5, :12]

## 2. Chạy Apriori và FP-Growth cùng điều kiện

Cả hai thuật toán dùng **cùng dữ liệu, cùng ngưỡng và cùng số item tối đa**. Vì vậy, nếu cùng luật được tìm thấy thì có thể so sánh thời gian chạy một cách công bằng.

In [ ]:
rules_by_algorithm = {}
summaries = []
for algorithm in ('apriori', 'fpgrowth'):
    rules, summary = mine_churn_rules(
        transactions, algorithm=algorithm, min_support=MIN_SUPPORT,
        min_confidence=MIN_CONFIDENCE, min_lift=MIN_LIFT, max_len=MAX_LEN
    )
    rules_by_algorithm[algorithm] = rules
    summaries.append(summary)

comparison = pd.DataFrame(summaries)
comparison

## 3. Xem các luật nổi bật

Chỉ có luật mà vế phải đúng là `Churn=Yes` mới được giữ lại. Không diễn giải luật là quan hệ nhân quả; đây chỉ là mối liên hệ trong dữ liệu.

In [ ]:
TOP_N = 10
rules_by_algorithm['apriori'].head(TOP_N)

## 4. Sinh và xem ảnh Lift

Thanh càng vượt xa đường Lift = 1 thì tổ hợp ở vế trái càng liên hệ mạnh hơn với `Churn=Yes`.

In [ ]:
figures_dir = REPORTS_DIR / 'figures' / 'association_rules'
for algorithm, rules in rules_by_algorithm.items():
    image_path = figures_dir / f'top_rules_lift_{algorithm}.png'
    save_top_rules_plot(rules, image_path, top_n=TOP_N)
    print(algorithm, '→', image_path)
    display(Image(filename=str(image_path)))